## **Importing the dependencies**

In [1]:
print("Nikhil, you are ready to go !")

Nikhil, you are ready to go !


In [2]:
# import os
# import numpy as np
# import nibabel as nib
# import torch
# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms
# from PIL import Image

# class LungCancerDataset(Dataset):
#     def __init__(self, root_dir):
#         self.image_paths = []
#         self.mask_paths = []
#         patient_dirs = sorted(os.listdir(root_dir))
#         for patient in patient_dirs:
#             data_dir = os.path.join(root_dir, patient, 'data')
#             mask_dir = os.path.join(root_dir, patient, 'masks')
            
#             for file_name in sorted(os.listdir(data_dir)):
#                 img_path = os.path.join(data_dir, file_name)
#                 mask_path = os.path.join(mask_dir, file_name)  # same name
#                 self.image_paths.append(img_path)
#                 self.mask_paths.append(mask_path)

#     def __len__(self):
#         return len(self.image_paths)

#     def __getitem__(self, idx):
#         image = np.load(self.image_paths[idx])
#         mask = np.load(self.mask_paths[idx])
#         image = torch.from_numpy(image).unsqueeze(0).float()
#         mask = torch.from_numpy(mask).unsqueeze(0).float()
        
#         return image, mask


# # Custom Dataset for Pneumonia (COVID) Dataset (2D images)
# class PneumoniaDataset(Dataset):
#     def __init__(self, ct_scan_dir, mask_dir, transform=None):
#         self.ct_scan_dir = ct_scan_dir
#         self.mask_dir = mask_dir
#         self.transform = transform
        
#         self.ct_scan_paths = [os.path.join(ct_scan_dir, file) for file in os.listdir(ct_scan_dir)]
#         self.mask_paths = [os.path.join(mask_dir, file) for file in os.listdir(mask_dir)]

#     def __len__(self):
#         return len(self.ct_scan_paths)

#     def __getitem__(self, idx):
#         # Load .nii files
#         ct_scan_nii = nib.load(self.ct_scan_paths[idx])
#         mask_nii = nib.load(self.mask_paths[idx])
        
#         # Extract 2D slices and normalize them
#         ct_scan = ct_scan_nii.get_fdata()
#         mask = mask_nii.get_fdata()
        
#         # Normalize the CT scan pixel values
#         ct_scan = ct_scan / np.max(ct_scan)
        
#         # Take a slice (let's assume the middle slice for simplicity)
#         slice_idx = ct_scan.shape[2] // 2
#         ct_scan_slice = ct_scan[:, :, slice_idx]
#         mask_slice = mask[:, :, slice_idx]
        
#         # Convert to Image format (2D, shape 256x256)
#         ct_scan_slice = Image.fromarray(ct_scan_slice)
#         ct_scan_slice = ct_scan_slice.resize((256, 256))
#         mask_slice = Image.fromarray(mask_slice)
#         mask_slice = mask_slice.resize((256, 256))
#         mask_slice = np.array(mask_slice)
#         mask_slice = np.where(mask_slice > 0, 2, 0)
        
#         # Apply transformations if available
#         if self.transform:
#             ct_scan_slice = self.transform(ct_scan_slice)
#             mask_slice = self.transform(mask_slice)

#         return ct_scan_slice, mask_slice

# # Combine both datasets into one
# class CombinedDataset(Dataset):
#     def __init__(self, lung_cancer_data_dir, pneumonia_ct_scan_dir, pneumonia_mask_dir, transform=None):
#         self.lung_cancer_dataset = LungCancerDataset(lung_cancer_data_dir)
#         self.pneumonia_dataset = PneumoniaDataset(pneumonia_ct_scan_dir, pneumonia_mask_dir, transform)
        
#     def __len__(self):
#         return len(self.lung_cancer_dataset) + len(self.pneumonia_dataset)
    
#     def __getitem__(self, idx):
#         if idx < len(self.lung_cancer_dataset):
#             return self.lung_cancer_dataset[idx]
#         else:
#             return self.pneumonia_dataset[idx - len(self.lung_cancer_dataset)]

# # Define transformations (optional)
# transform = transforms.Compose([
#     transforms.ToTensor(),
# ])

# # Define dataset directories
# lung_cancer_train_dir = "/kaggle/input/lung-cancer-segment/train"
# lung_cancer_val_dir = "/kaggle/input/lung-cancer-segment/val"
# pneumonia_ct_scan_dir = "/kaggle/input/covid19-ct-scans/ct_scans"
# pneumonia_mask_dir = "/kaggle/input/covid19-ct-scans/infection_mask"

# # Create Combined Dataset and DataLoader
# dataset = CombinedDataset(lung_cancer_train_dir, pneumonia_ct_scan_dir, pneumonia_mask_dir, transform)
# dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# # Now, you can loop through the DataLoader for your model training
# for images, masks in dataloader:
#     # Your training code here
#     print(images.shape, masks.shape)  # This will print (batch_size, channels, 256, 256)
#     break

In [3]:
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
import torchvision.transforms as transforms
from PIL import Image

class LungSegmentationDataset(Dataset):
    def __init__(self, root_dir):
        self.image_paths = []
        self.mask_paths = []
        patient_dirs = sorted(os.listdir(root_dir))
        for patient in patient_dirs:
            data_dir = os.path.join(root_dir, patient, 'data')
            mask_dir = os.path.join(root_dir, patient, 'masks')
            
            for file_name in sorted(os.listdir(data_dir)):
                img_path = os.path.join(data_dir, file_name)
                mask_path = os.path.join(mask_dir, file_name)  # same name
                self.image_paths.append(img_path)
                self.mask_paths.append(mask_path)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.load(self.image_paths[idx])
        mask = np.load(self.mask_paths[idx])
        image = torch.from_numpy(image).unsqueeze(0).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image, mask

In [4]:
import os
import numpy as np
import nibabel as nib
import cv2
import shutil

def process_and_save_data(ct_scan_dir, mask_dir, output_dir, train_range, val_range, num_slices_to_remove=60):
    """
    Process CT scan and mask data, normalize, resize, remove slices, and save to npy files.
    
    Args:
    - ct_scan_dir: Directory containing CT scan images (.nii files).
    - mask_dir: Directory containing infection masks (.nii files).
    - output_dir: Directory to save the processed data.
    - train_range: Range of files for training (start, end).
    - val_range: Range of files for validation (start, end).
    - num_slices_to_remove: Number of slices to remove from lower abdomen to neck area.
    """
    
    # Create output directories if they don't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Process and save train data
    for i in range(train_range[0], train_range[1] + 1):
        ct_file = os.path.join(ct_scan_dir, f'coronacases_org_{i:03d}.nii')
        mask_file = os.path.join(mask_dir, f'coronacases_{i:03d}.nii')
        
        # Load the CT scan and mask
        ct_scan = nib.load(ct_file).get_fdata()
        mask = nib.load(mask_file).get_fdata()
        
        # Remove the top 20 slices (lower abdomen to neck)
        ct_scan = ct_scan[num_slices_to_remove:, :, :]
        mask = mask[num_slices_to_remove:, :, :]
        
        # Normalize CT scan by dividing by the maximum value
        ct_scan = ct_scan / np.max(ct_scan)
        
        # Resize each slice to 256x256 using nearest-neighbor interpolation
        ct_scan_resized = np.array([cv2.resize(ct_scan[:,:,i], (256, 256), interpolation=cv2.INTER_NEAREST) for i in range(ct_scan.shape[2])])
        mask_resized = np.array([cv2.resize(mask[:,:,i], (256, 256), interpolation=cv2.INTER_NEAREST) for i in range(mask.shape[2])])
        
        # Save as npy files
        for slice_idx in range(ct_scan_resized.shape[0]):
            os.makedirs(os.path.join(output_dir, 'train',str(i), 'data'), exist_ok=True)
            os.makedirs(os.path.join(output_dir, 'train', str(i),'masks'), exist_ok=True)
            np.save(os.path.join(output_dir, 'train', str(i),'data', f'{slice_idx}.npy'), ct_scan_resized[slice_idx,:,:])
            np.save(os.path.join(output_dir, 'train', str(i),'masks', f'{slice_idx}.npy'), mask_resized[slice_idx,:,:])
    
    # Process and save validation data
    for i in range(val_range[0], val_range[1] + 1):
        ct_file = os.path.join(ct_scan_dir, f'coronacases_org_{i:03d}.nii')
        mask_file = os.path.join(mask_dir, f'coronacases_{i:03d}.nii')
        
        # Load the CT scan and mask
        ct_scan = nib.load(ct_file).get_fdata()
        mask = nib.load(mask_file).get_fdata()
        
        # Remove the top 20 slices (lower abdomen to neck)
        ct_scan = ct_scan[num_slices_to_remove:, :, :]
        mask = mask[num_slices_to_remove:, :, :]
        
        # Normalize CT scan by dividing by the maximum value
        ct_scan = ct_scan / np.max(ct_scan)
        
        # Resize each slice to 256x256 using nearest-neighbor interpolation
        ct_scan_resized = np.array([cv2.resize(ct_scan[:,:,i], (256, 256), interpolation=cv2.INTER_NEAREST) for i in range(ct_scan.shape[2])])
        mask_resized = np.array([cv2.resize(mask[:,:,i], (256, 256), interpolation=cv2.INTER_NEAREST) for i in range(mask.shape[2])])
        
        # Save as npy files
        for slice_idx in range(ct_scan_resized.shape[0]):
            os.makedirs(os.path.join(output_dir, 'val',str(i), 'data'), exist_ok=True)
            os.makedirs(os.path.join(output_dir, 'val',str(i), 'masks'), exist_ok=True)
            np.save(os.path.join(output_dir, 'val', str(i),'data', f'{slice_idx}.npy'), ct_scan_resized[slice_idx,:,:])
            np.save(os.path.join(output_dir, 'val', str(i),'masks', f'{slice_idx}.npy'), mask_resized[slice_idx,:,:])

# Define input and output directories
ct_scan_dir = '/kaggle/input/covid19-ct-scans/ct_scans'
mask_dir = '/kaggle/input/covid19-ct-scans/infection_mask'
output_dir = 'Pneumonia'

# Define the ranges for training and validation
train_range = (1, 8)  # Train: coronacases_001.nii to coronacases_008.nii
val_range = (9, 10)   # Validation: coronacases_009.nii to coronacases_010.nii

# Process the dataset
process_and_save_data(ct_scan_dir, mask_dir, output_dir, train_range, val_range)


In [5]:
from warnings import filterwarnings

filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import os
import gc
import random
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import interact

from tqdm.auto import tqdm
import torch.nn.functional as F
from torchvision.transforms.v2 import GaussianNoise
from torchmetrics import JaccardIndex, Precision, Recall, Specificity, F1Score, AUROC
import torch.optim as optim

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr_skimage
from skimage.metrics import mean_squared_error as mse_skimage
from skimage.metrics import hausdorff_distance
from scipy.ndimage import distance_transform_edt
from tensorflow.keras.preprocessing.image import load_img
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


import torch

def Dice(preds, targets, smooth=1e-6, threshold=None):
    
    if preds.shape != targets.shape:
        raise ValueError("Predictions and targets must have the same shape.")
    
    # Apply thresholding for binary or multi-class case
    if threshold is not None:
        preds = (preds > threshold).float()

    # Flatten tensors except for batch and channel dimensions
    preds = preds.flatten(2)  # (B, C, H*W)
    targets = targets.flatten(2)

    intersection = (preds * targets).sum(dim=-1)
    union = preds.sum(dim=-1) + targets.sum(dim=-1)

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice.mean()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

2025-06-25 18:22:26.753785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750875746.938031      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750875746.992625      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
# import os
# import numpy as np
# from torch.utils.data import Dataset, DataLoader
# import torch
# import torchvision.transforms as transforms
# from PIL import Image

# class LungSegmentationDataset(Dataset):
#     def __init__(self, root_dir):
#         self.image_paths = []
#         self.mask_paths = []
#         patient_dirs = sorted(os.listdir(root_dir))
#         for patient in patient_dirs:
#             data_dir = os.path.join(root_dir, patient, 'data')
#             mask_dir = os.path.join(root_dir, patient, 'masks')
            
#             for file_name in sorted(os.listdir(data_dir)):
#                 img_path = os.path.join(data_dir, file_name)
#                 mask_path = os.path.join(mask_dir, file_name)  # same name
#                 self.image_paths.append(img_path)
#                 self.mask_paths.append(mask_path)

#     def __len__(self):
#         return len(self.image_paths)

#     def __getitem__(self, idx):
#         image = np.load(self.image_paths[idx])
#         mask = np.load(self.mask_paths[idx])
#         image = torch.from_numpy(image).unsqueeze(0).float()
#         mask = torch.from_numpy(mask).unsqueeze(0).float()
        
#         return image, mask

# train_dataset = LungSegmentationDataset(root_dir='/kaggle/input/lung-cancer-segment/train')

# val_dataset = LungSegmentationDataset(root_dir='/kaggle/input/lung-cancer-segment/val')

# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
# val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=4)


In [7]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random

class CombinedLungDataset(Dataset):
    def __init__(self, lung_cancer_dir, pneumonia_dir):
        self.lung_cancer_dataset = LungSegmentationDataset(lung_cancer_dir)  # for lung cancer data
        self.pneumonia_dataset = LungSegmentationDataset(pneumonia_dir)  # for pneumonia data
        
        # Combine both datasets
        self.all_data = []
        
        # Add lung cancer data with label 1
        for img_path, mask_path in zip(self.lung_cancer_dataset.image_paths, self.lung_cancer_dataset.mask_paths):
            self.all_data.append((img_path, mask_path, 1))  # 1 for lung cancer
        
        # Add pneumonia data with label 2
        for img_path, mask_path in zip(self.pneumonia_dataset.image_paths, self.pneumonia_dataset.mask_paths):
            self.all_data.append((img_path, mask_path, 2))  # 2 for pneumonia

    def __len__(self):
        return len(self.all_data)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.all_data[idx]
        
        # Load image and mask
        image = np.load(img_path)
        mask = np.load(mask_path)
        
        # Label the dataset (lung cancer -> 1, pneumonia -> 2)
        mask = np.where(mask > 0, label, 0)  # Mask stays 0 for background, and labeled as 1 or 2
        
        # Convert to torch tensors
        image = torch.from_numpy(image).unsqueeze(0).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image, mask

# Paths to the datasets
lung_cancer_dir_train = '/kaggle/input/lung-cancer-segment/train'
pneumonia_dir_train = '/kaggle/working/Pneumonia/train'
train_dataset = CombinedLungDataset(lung_cancer_dir_train, pneumonia_dir_train)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


lung_cancer_dir_val = '/kaggle/input/lung-cancer-segment/val'
pneumonia_dir_val = '/kaggle/working/Pneumonia/val'
val_dataset = CombinedLungDataset(lung_cancer_dir_val, pneumonia_dir_val)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

len(train_loader), len(val_loader)


(2057, 1892)

## **Importing Libraries for UNet**

In [8]:
import random
from tqdm import tqdm
import csv
import time 

import torch
import torch.nn as nn
from torchvision import models
from torch.nn.functional import relu
import torch.nn.functional as F

## **UNet Architecture**

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet1(nn.Module):
    def __init__(self, n_class=3):
        super().__init__()

        self.e11 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.e12 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e21 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.e22 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e31 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.e32 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e41 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.e42 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e51 = nn.Conv2d(512, 1024, kernel_size=3, padding=1)
        self.e52 = nn.Conv2d(1024, 1024, kernel_size=3, padding=1)

        self.upconv1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.d11 = nn.Conv2d(1024, 512, kernel_size=3, padding=1)
        self.d12 = nn.Conv2d(512, 512, kernel_size=3, padding=1)

        self.upconv2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.d21 = nn.Conv2d(512, 256, kernel_size=3, padding=1)
        self.d22 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.d31 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.d32 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        self.upconv4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.d41 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.d42 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.outconv = nn.Conv2d(64, n_class, kernel_size=1)    # For Segmentation mask
        self.finalconv = nn.Conv2d(64, n_class, kernel_size=1)   # For Denoised image

    def forward(self, x):
        x = F.relu(self.e11(x))
        x1 = F.relu(self.e12(x))
        x = self.pool1(x1)

        x = F.relu(self.e21(x))
        x2 = F.relu(self.e22(x))
        x = self.pool2(x2)

        x = F.relu(self.e31(x))
        x3 = F.relu(self.e32(x))
        x = self.pool3(x3)

        x = F.relu(self.e41(x))
        x4 = F.relu(self.e42(x))
        x = self.pool4(x4)

        x = F.relu(self.e51(x))
        x = F.relu(self.e52(x))

        x = self.upconv1(x)
        x = torch.cat([x, x4], dim=1)
        x = F.relu(self.d11(x))
        x = F.relu(self.d12(x))

        x = self.upconv2(x)
        x = torch.cat([x, x3], dim=1)
        x = F.relu(self.d21(x))
        x = F.relu(self.d22(x))

        x = self.upconv3(x)
        x = torch.cat([x, x2], dim=1)
        x = F.relu(self.d31(x))
        x = F.relu(self.d32(x))

        x = self.upconv4(x)
        x = torch.cat([x, x1], dim=1)
        x = F.relu(self.d41(x))
        x = F.relu(self.d42(x))

        out = self.outconv(x) 
        return out


# UNet Segmentation

In [10]:
def seg_train_model(model, train_loader, val_loader, device, name, num_epochs=50):
    save_dir = name
    os.makedirs(save_dir, exist_ok=True)

    # Losses
    cross_entropy_loss = nn.CrossEntropyLoss()  # For multiclass segmentation

    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Log DataFrame with custom column order
    columns = [
        'Epoch', 'Total Loss', 'Dice Score', 'Time (s)',
        'Precision', 'Recall', 'F1 Score', 'Specificity', 
        'Val Total Loss',  'Val Dice Score',  'Val Time (s)', 
        'Val Precision', 'Val Recall', 'Val F1 Score', 'Val Specificity',
    ]
    log_df = pd.DataFrame(columns=columns)

    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        metrics = {col: 0.0 for col in columns}  # Initialize metrics dict

        for phase in ['train', 'val']:
            dataloader = train_loader if phase == 'train' else val_loader
            model.train() if phase == 'train' else model.eval()

            batch_metrics = {
                'total_loss': [],
                'dice': [], 'precision': [], 'recall': [], 'specificity': [],
            }

            start_time = time.time()

            with torch.set_grad_enabled(phase == 'train'):
                for noisy_img, mask in tqdm(dataloader, desc=f"{phase.capitalize()} Epoch {epoch+1}", leave=False):
                    noisy_img = noisy_img.to(device)
                    mask = mask.to(device)

                    # Remove singleton dimension from mask if it's (batch_size, 1, height, width)
                    mask = mask.squeeze(1)  # Shape should now be (batch_size, height, width)
                    mask = mask.long()  # Convert to Long type (required by CrossEntropyLoss)

                    # Forward pass
                    seg_mask_logits = model(noisy_img)

                    # Losses
                    loss_seg = cross_entropy_loss(seg_mask_logits, mask)
                    total_loss = loss_seg

                    if phase == 'train':
                        optimizer.zero_grad()
                        total_loss.backward()
                        optimizer.step()

                    # Get predictions by taking argmax across the class dimension
                    seg_probs = torch.argmax(seg_mask_logits, dim=1)  # Shape: (batch_size, height, width)

                    # Segmentation metrics
                    dice = dice_score(seg_probs, mask).item()
                    prec = precision(seg_probs, mask).item()
                    rec = recall(seg_probs, mask).item()
                    spec = specificity(seg_probs, mask).item()
                    f1 = f1_score(prec, rec)

                    # Append batch metrics
                    batch_metrics['total_loss'].append(total_loss.item())
                    batch_metrics['dice'].append(dice)
                    batch_metrics['precision'].append(prec)
                    batch_metrics['recall'].append(rec)
                    batch_metrics['specificity'].append(spec)

            # Aggregate metrics
            end_time = time.time()
            avg = {k: np.mean(v) for k, v in batch_metrics.items()}
            time_taken = end_time - start_time

            # Update metrics dict
            prefix = '' if phase == 'train' else 'Val '
            metrics[f'{prefix}Total Loss'] = avg['total_loss']
            metrics[f'{prefix}Dice Score'] = avg['dice']
            metrics[f'{prefix}Time (s)'] = time_taken
            metrics[f'{prefix}Precision'] = avg['precision']
            metrics[f'{prefix}Recall'] = avg['recall']
            metrics[f'{prefix}F1 Score'] = f1_score(avg['precision'], avg['recall'])
            metrics[f'{prefix}Specificity'] = avg['specificity']

        # Append to log
        log_df = pd.concat([log_df, pd.DataFrame([metrics])], ignore_index=True)

        # Save best model
        if metrics['Val Total Loss'] < best_val_loss:
            best_val_loss = metrics['Val Total Loss']
            best_model_path = os.path.join(save_dir, f'{name}_best_val_loss.pt')
            torch.save(model.state_dict(), best_model_path)

        # Print progress
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {metrics['Total Loss']:.4f} | "
              f"Train Dice: {metrics['Dice Score']:.4f} | "
              f"Val Loss: {metrics['Val Total Loss']:.4f} | "
              f"Val Dice: {metrics['Val Dice Score']:.4f} | "
              )

    # Save logs
    log_path = os.path.join(save_dir, f'{name}_Training.csv')
    log_df.to_csv(log_path, index=False, float_format='%.4f')
    print(f"Training complete. Logs saved to {log_path}")

    return model


In [11]:
import torch
import torch.nn as nn
import pandas as pd
import time
from tqdm import tqdm

# -------------------------------
# 📊 Metric Functions (for multiclass)
# -------------------------------

def dice_score(preds, targets, num_classes=3, smooth=1e-6):
    dice = 0.0
    for i in range(num_classes):
        preds_i = (preds == i).float()
        targets_i = (targets == i).float()
        intersection = (preds_i * targets_i).sum()
        total = preds_i.sum() + targets_i.sum()
        dice += (2.0 * intersection + smooth) / (total + smooth)
    return dice / num_classes

def precision(preds, targets, num_classes=3, smooth=1e-6):
    prec = 0.0
    for i in range(num_classes):
        preds_i = (preds == i).float()
        targets_i = (targets == i).float()
        tp = (preds_i * targets_i).sum()
        fp = preds_i.sum() - tp
        prec += (tp + smooth) / (tp + fp + smooth)
    return prec / num_classes

def recall(preds, targets, num_classes=3, smooth=1e-6):
    rec = 0.0
    for i in range(num_classes):
        preds_i = (preds == i).float()
        targets_i = (targets == i).float()
        tp = (preds_i * targets_i).sum()
        fn = targets_i.sum() - tp
        rec += (tp + smooth) / (tp + fn + smooth)
    return rec / num_classes

def specificity(preds, targets, num_classes=3, smooth=1e-6):
    spec = 0.0
    for i in range(num_classes):
        preds_i = (preds == i).float()
        targets_i = (targets == i).float()
        tn = ((1 - preds_i) * (1 - targets_i)).sum()
        fp = preds_i.sum() - (preds_i * targets_i).sum()
        spec += (tn + smooth) / (tn + fp + smooth)
    return spec / num_classes

def f1_score(precision, recall, beta=1.0, smooth=1e-6):
    return (1 + beta**2) * (precision * recall + smooth) / (beta**2 * precision + recall + smooth)

# -------------------------------
# 🧠 Testing Loop
# -------------------------------

def seg_test_model(model, test_loader, device, name):
    save_dir = name
    os.makedirs(save_dir, exist_ok=True)

    # Loss function for multiclass segmentation
    cross_entropy_loss = nn.CrossEntropyLoss()

    # Metrics handler

    # Column order for metrics
    columns = ['Total Loss','Dice Score', 'Time (s)',
               'Precision', 'Recall', 'F1 Score', 'Specificity']

    # Initialize metrics dictionary
    metrics = {col: 0.0 for col in columns}

    # Batch metrics
    batch_metrics = {'total_loss': [],'dice': [], 'precision': [], 'recall': [], 'specificity': []}

    model.eval()  # Set model to evaluation mode
    start_time = time.time()

    with torch.no_grad():
        for noisy_img, mask in tqdm(test_loader, desc=f"TEST-Epoch", leave=False):
            noisy_img = noisy_img.to(device)
            mask = mask.to(device)

            # Forward pass (Logits for each class)
            seg_mask_logits = model(noisy_img)
            mask = mask.squeeze(1)  # Shape should now be (batch_size, height, width)
            mask = mask.long()
            # Loss
            loss_seg = cross_entropy_loss(seg_mask_logits, mask)
            total_loss = loss_seg

            # Get predictions by taking argmax across the class dimension
            seg_probs = torch.argmax(seg_mask_logits, dim=1)  # Shape: (batch_size, height, width)

            # Segmentation metrics (multiclass)
            dice = dice_score(seg_probs, mask).item()
            prec = precision(seg_probs, mask).item()
            rec = recall(seg_probs, mask).item()
            spec = specificity(seg_probs, mask).item()
            f1 = f1_score(prec, rec)

            # Append batch metrics
            batch_metrics['total_loss'].append(total_loss.item())
            batch_metrics['dice'].append(dice)
            batch_metrics['precision'].append(prec)
            batch_metrics['recall'].append(rec)
            batch_metrics['specificity'].append(spec)

    # Aggregate metrics
    avg = {k: np.mean(v) for k, v in batch_metrics.items()}
    time_taken = time.time() - start_time

    # Fill metrics dictionary (test phase)
    metrics.update({
        'Dice Score': avg['dice'],
        'Time (s)': time_taken,
        'Precision': avg['precision'],
        'Recall': avg['recall'],
        'F1 Score': f1_score(avg['precision'], avg['recall']),
        'Specificity': avg['specificity'],
    })

    # Create DataFrame and save
    log_df = pd.DataFrame([metrics])
    log_path = os.path.join(save_dir, f'{name}_Testing.csv')
    log_df.to_csv(log_path, index=False, float_format='%.4f')

    print(f"Test complete. Results saved to {log_path}")
    print(f"Test Dice: {avg['dice']:.4f}")

    return log_df


In [12]:
# -------------------------------
# Parameters
# -------------------------------
depth = [1, 1, 1, 1, 1, 1]
NUM_CLASS = 3  
NUM_EPOCHS = 30
multi_tasks_model_dict = {
    'U-Net': UNet1(n_class=NUM_CLASS)  # Updated to handle multiclass segmentation
}

# -------------------------------
# Training and Testing Loop
# -------------------------------
for name, model in multi_tasks_model_dict.items():
    model = model.to(device)
    
    # Training the model
    print(f"Training {name} model...")
    trained_model = seg_train_model(
        model, train_loader=train_loader, val_loader=val_loader,
        device=device, name=name, num_epochs=NUM_EPOCHS
    )
    
    # Testing the model
    print(f"Testing {name} model...")
    test_results = seg_test_model(trained_model, val_loader, device, name=name)
    
    print(f"---------------------------{name} training and testing is completed--------------------------------------")

Training U-Net model...


Epoch 1/30 | Train Loss: 0.0231 | Train Dice: 0.6747 | Val Loss: 0.0129 | Val Dice: 0.9286 | 


Epoch 2/30 | Train Loss: 0.0087 | Train Dice: 0.6847 | Val Loss: 0.0109 | Val Dice: 0.9286 | 


Epoch 3/30 | Train Loss: 0.0074 | Train Dice: 0.6857 | Val Loss: 0.0099 | Val Dice: 0.9286 | 


Epoch 4/30 | Train Loss: 0.0067 | Train Dice: 0.6802 | Val Loss: 0.0107 | Val Dice: 0.9286 | 


...

Epoch 27/30 | Train Loss: 0.0007 | Train Dice: 0.9363 | Val Loss: 0.0183 | Val Dice: 0.9416 | 


Epoch 28/30 | Train Loss: 0.0007 | Train Dice: 0.9370 | Val Loss: 0.0167 | Val Dice: 0.9422 | 


Epoch 29/30 | Train Loss: 0.0007 | Train Dice: 0.9408 | Val Loss: 0.0224 | Val Dice: 0.9299 | 


Epoch 30/30 | Train Loss: 0.0007 | Train Dice: 0.9383 | Val Loss: 0.0185 | Val Dice: 0.9359 | 
Training complete. Logs saved to U-Net/U-Net_Training.csv
Testing U-Net model...


Test complete. Results saved to U-Net/U-Net_Testing.csv
Test Dice: 0.9359
---------------------------U-Net training and testing is completed--------------------------------------


In [13]:
StoptheTraining

NameError: name 'StoptheTraining' is not defined

In [ ]:
# test_model = MT_UNet_YOLO_Large([1, 64, 128, 256, 512, 512], [1,1,1,1,1,1], [True, True], 1).to(device)
# test_model.load_state_dict(torch.load('/kaggle/input/you-net-lung-cancer/pytorch/default/1/YOU-Net_best_val_loss.pt'))

In [ ]:
import glob

path ='/kaggle/input/lung-cancer-segment'
val_data_path = os.path.join(path, 'val', '57', 'data')
val_mask_path = os.path.join(path, 'val', '57', 'masks')
image_files = sorted(glob.glob(os.path.join(val_data_path, "*.npy")))
mask_files = sorted(glob.glob(os.path.join(val_mask_path, "*.npy")))
all_images = [np.load(f) for f in image_files]
all_masks = [np.load(f) for f in mask_files]

image_volume = np.stack(all_images, axis=0) # Stack along a new axis (depth)
mask_volume = np.stack(all_masks, axis=0)   # Stack along a new axis (depth)

predicted_masks_list = []
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
with torch.no_grad():
    for i in tqdm(range(image_volume.shape[0]), desc="Processing slices"):
        img_slice = image_volume[i, :, :]
        img_tensor = torch.from_numpy(img_slice).unsqueeze(0).unsqueeze(0).float().to(device) # Add batch and channel
        output = model(img_tensor)
        predicted_mask_slice = torch.sigmoid(output).squeeze().cpu().numpy() > 0.5 # Remove batch and channel, move to cpu, convert to numpy, threshold
        predicted_masks_list.append(predicted_mask_slice)

predicted_mask_volume = np.stack(predicted_masks_list, axis=0)

In [ ]:
imgTarget = image_volume.transpose(1,2,0)
imgMask = mask_volume.transpose(1,2,0)
predImg = predicted_mask_volume.transpose(1,2,0)

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import measure


In [ ]:
vertices, faces, _, _ = measure.marching_cubes(predImg, level=0.5)
ctvertices, ctfaces, _, _ = measure.marching_cubes(imgTarget, level=0.5)
maskvertices, maskfaces, _, _ = measure.marching_cubes(imgMask, level=0.5)

# Optional: Display the number of vertices and faces for debugging
print(f"Inferred Mask: {len(vertices)} vertices, {len(faces)} faces")
print(f"Input Scan: {len(ctvertices)} vertices, {len(ctfaces)} faces")
print(f"Annotated Mask: {len(maskvertices)} vertices, {len(maskfaces)} faces")

In [ ]:
import os
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F

# Define paths
dataInputPath = '/kaggle/input/medical-segmentation-decathlon-lung'
imagePathInput = os.path.join(dataInputPath, 'imagesTr/')
maskPathInput = os.path.join(dataInputPath, 'labelsTr/')

targetImageFile = 'lung_003.nii'
targetMaskFile = 'lung_003.nii'

targetImagePath = os.path.join(imagePathInput, targetImageFile)
targetMaskPath = os.path.join(maskPathInput, targetMaskFile)

def preprocess_image_and_mask(img_path, mask_path):
    # Load NIfTI files
    img_nii = nib.load(img_path).get_fdata().astype(np.float32)
    lbl_nii = nib.load(mask_path).get_fdata().astype(np.int64)

    # Ensure shape match
    if img_nii.shape != lbl_nii.shape:
        raise ValueError("Image and mask shapes do not match.")

    images_list = []
    mask_list= []

    image_data = np.zeros((256, 256, img_nii.shape[2]))

    mask_data= np.zeros((256, 256, img_nii.shape[2]))

    for i in range(img_nii.shape[2]):
        img_slice = img_nii[:, :, i]
        lbl_slice = lbl_nii[:, :, i]


        # Normalize image to [0,1]
        img_slice = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-8)

        # Convert to tensor and add channel dim
        img_tensor = torch.from_numpy(img_slice).unsqueeze(0)  # (1, H, W)
        mask_tensor = torch.from_numpy(lbl_slice).unsqueeze(0).float()  # (1, H, W)

        # Resize to (256, 256)
        img_tensor = F.interpolate(img_tensor.unsqueeze(0), size=(256, 256), mode='bilinear', align_corners=False).squeeze(0)
        mask_tensor = F.interpolate(mask_tensor.unsqueeze(0), size=(256, 256), mode='nearest').squeeze(0)
        images_list.append(img_tensor)
        mask_list.append(mask_tensor)
        image_data[:,:,i] = img_tensor
        mask_data[:,:,i] = mask_tensor
        # all_slices.append((img_tensor, mask_tensor))

    return image_data, mask_data

# Run preprocessing
imgTarget, imgMask = preprocess_image_and_mask(targetImagePath, targetMaskPath)


In [ ]:
predictions = np.zeros((256, 256, 288))

with torch.no_grad():
    for idx in range(imgTarget.shape[-1]):
        img1 = imgTarget[:, :, idx][np.newaxis, np.newaxis, :, :] 
        img_tensor = torch.from_numpy(img1).float().cuda()
        out = test_model(img_tensor)
        prediction_slice = out.squeeze().cpu() 
        predictions[:, :, idx] = (torch.sigmoid(prediction_slice).numpy()>0.5).astype(np.int64)

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
# from stl import mesh
from skimage import measure

predImg = predictions

# Generate vertices and faces using marching cubes
vertices, faces, _, _ = measure.marching_cubes(predImg, level=0.5)
ctvertices, ctfaces, _, _ = measure.marching_cubes(imgTarget, level=0.5)
maskvertices, maskfaces, _, _ = measure.marching_cubes(imgMask, level=0.5)

# Optional: Display the number of vertices and faces for debugging
print(f"Inferred Mask: {len(vertices)} vertices, {len(faces)} faces")
print(f"Input Scan: {len(ctvertices)} vertices, {len(ctfaces)} faces")
print(f"Annotated Mask: {len(maskvertices)} vertices, {len(maskfaces)} faces")

In [ ]:
# prompt: calculate the dice score of predImg and imgMask

def calculate_dice_score(predImg, imgMask):
    """
    Calculates the Dice Similarity Coefficient (DSC) between two binary masks.

    Args:
        predImg (np.ndarray): Predicted binary mask.
        imgMask (np.ndarray): Ground truth binary mask.

    Returns:
        float: Dice score.
    """
    # Ensure both inputs are binary (0 or 1)
    predImg_binary = (predImg > 0.5).astype(np.float32)
    imgMask_binary = (imgMask > 0.5).astype(np.float32)

    intersection = np.sum(predImg_binary * imgMask_binary)
    sum_masks = np.sum(predImg_binary) + np.sum(imgMask_binary)

    # Avoid division by zero
    if sum_masks == 0:
        return 1.0  # Or 0.0, depending on how you define Dice for empty masks

    dice = (2. * intersection) / sum_masks
    return dice

# Calculate and print the Dice score
dice_score = calculate_dice_score(predImg, imgMask)
print(f"Dice Score: {dice_score}")

In [ ]:
!pip install numpy-stl -q

In [ ]:
from stl import mesh

def dataToMesh(vert, faces):
    stl_mesh = mesh.Mesh(np.zeros(faces.shape[0], dtype=mesh.Mesh.dtype))
    for i, f in enumerate(faces):
        for j in range(3):
            stl_mesh.vectors[i][j] = vert[f[j], :]
    return stl_mesh

# Convert and save all .stl files
output_path = './'  # Define your output directory
inference_mesh = dataToMesh(vertices, faces)
inference_mesh.save(output_path + 'Inferenced_lung.stl')

# input_mesh = dataToMesh(ctvertices, ctfaces)
# input_mesh.save(output_path + 'Input_lung_003.stl')

mask_mesh = dataToMesh(maskvertices, maskfaces)
mask_mesh.save(output_path + 'Mask_lung.stl')

print("STL files saved successfully.")

In [ ]:
import open3d as o3d

stl_file_path = "/content/Inferenced_lung_003.stl"  
mesh = o3d.io.read_triangle_mesh(stl_file_path)
print("_")
pointcloud = mesh.sample_points_poisson_disk(100000)
xyz_load = np.asarray(pointcloud.points, dtype=np.float32)
print('xyz_load shape', xyz_load.shape)